In [ ]:
# Install dependencies
!pip install -q tensorflow
!wget -q http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
!unzip -q cornell_movie_dialogs_corpus.zip



In [ ]:
# Load and clean
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9?.!,]+", " ", text)
    return text.strip()

lines = {}
with open("cornell movie-dialogs corpus/movie_lines.txt", encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = line.strip().split(" +++$+++ ")
        if len(parts) == 5:
            lines[parts[0]] = clean_text(parts[4])

pairs = []
with open("cornell movie-dialogs corpus/movie_conversations.txt", encoding='utf-8', errors='ignore') as f:
    for line in f:
        ids = eval(line.strip().split(" +++$+++ ")[-1])
        for i in range(len(ids) - 1):
            if ids[i] in lines and ids[i+1] in lines:
                pairs.append((lines[ids[i]], lines[ids[i+1]]))
        if len(pairs) >= 5000:
            break


In [ ]:
# Tokenize and pad
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

questions, answers = zip(*pairs)
questions = list(questions)  # ← FIX here
answers = ['startseq ' + ans + ' endseq' for ans in answers]


tokenizer = Tokenizer(num_words=8000, filters='')
tokenizer.fit_on_texts(questions + answers)

VOCAB_SIZE = len(tokenizer.word_index) + 1
MAX_LEN = 20

encoder_input = pad_sequences(tokenizer.texts_to_sequences(questions), maxlen=MAX_LEN, padding='post')
decoder_input = pad_sequences(tokenizer.texts_to_sequences(answers), maxlen=MAX_LEN, padding='post')

decoder_target = np.zeros_like(decoder_input)
decoder_target[:, :-1] = decoder_input[:, 1:]
decoder_target = np.expand_dims(decoder_target, -1)


In [ ]:
# Build the model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

EMBEDDING_DIM = 128
LATENT_DIM = 128

enc_inputs = Input(shape=(MAX_LEN,))
enc_embed = Embedding(VOCAB_SIZE, EMBEDDING_DIM)(enc_inputs)
_, state_h, state_c = LSTM(LATENT_DIM, return_state=True)(enc_embed)
enc_states = [state_h, state_c]

dec_inputs = Input(shape=(MAX_LEN,))
dec_embed = Embedding(VOCAB_SIZE, EMBEDDING_DIM)(dec_inputs)
dec_lstm = LSTM(LATENT_DIM, return_sequences=True)(dec_embed, initial_state=enc_states)
output = Dense(VOCAB_SIZE, activation='softmax')(dec_lstm)

model = Model([enc_inputs, dec_inputs], output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 128)   │  1,388,416 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 20, 128)   │  1,388,416 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │    131,584 │ embedding[0][0]   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 20, 128)   │    131,584 │ embedding_1[0][0… │
│                     │                   │            │ lstm[0][1],       │
│                     │                   │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 20, 10847) │  1,399,263 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,439,263 (16.93 MB)

 Trainable params: 4,439,263 (16.93 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train (fast, 10 epochs)
model.fit([encoder_input, decoder_input], decoder_target, batch_size=16, epochs=100)



Epoch 1/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.1669
Epoch 2/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.2512
Epoch 3/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.2030
Epoch 4/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.1711
Epoch 5/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.1420
Epoch 6/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.1337
Epoch 7/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.1249
Epoch 8/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.1226
Epoch 9/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.1226
Epoch 10/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.1207
Epoch 11/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.1214
Epoch 12/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.1228
Epoch 13/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.1270
Epoch 14/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.1119
Epoch 15/100
313/313 ━━━━━━━━

In [ ]:
def decode_sequence(input_text):
    input_text = clean_text(input_text)
    input_seq = pad_sequences(tokenizer.texts_to_sequences([input_text]), maxlen=MAX_LEN, padding='post')

    decoded_sentence = []
    target_seq = [tokenizer.word_index['startseq']]
    for _ in range(MAX_LEN):
        target_seq_padded = pad_sequences([target_seq], maxlen=MAX_LEN, padding='post')
        predictions = model.predict([input_seq, target_seq_padded], verbose=0)

        next_word_id = np.argmax(predictions[0, len(decoded_sentence)])
        next_word = tokenizer.index_word.get(next_word_id, '')

        if next_word == 'endseq' or next_word == '':
            break

        decoded_sentence.append(next_word)
        target_seq.append(next_word_id)

    return ' '.join(decoded_sentence)



In [ ]:
!pip install -q nltk
import nltk
nltk.download('punkt_tab')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

smoothie = SmoothingFunction().method4

def evaluate_bleu(num_samples=100):
    total_score = 0.0
    count = 0
    for i in range(num_samples):
        input_text = questions[i]
        target_text = answers[i].replace('startseq ', '').replace(' endseq', '')
        predicted = decode_sequence(input_text)

        reference = [word_tokenize(target_text)]
        hypothesis = word_tokenize(predicted)

        score = sentence_bleu(reference, hypothesis, smoothing_function=smoothie)
        total_score += score
        count += 1

        if i % 20 == 0:
            print(f"Input: {input_text}")
            print(f"Target: {target_text}")
            print(f"Predicted: {predicted}")
            print(f"BLEU: {score:.4f}\n")

    print(f"\nAverage BLEU Score on {num_samples} samples: {total_score / count:.4f}")
evaluate_bleu(100)  # evaluate on 100 samples


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Input: can we make this quick? roxanne korrine and andrew barrett are having an incredibly horrendous public break up on the quad. again.
Target: well, i thought we d start with pronunciation, if that s okay with you.
Predicted: well, i thought we d start with pronunciation, if that s okay with you.
BLEU: 1.0000

Input: i really, really, really wanna go, but i can t. not unless my sister goes.
Target: i m workin on it. but she doesn t seem to be goin for him.
Predicted: i m workin on it. but she doesn t seem to be goin for him.
BLEU: 1.0000

Input: no
Target: okay you re gonna need to learn how to lie.
Predicted: okay you re gonna need to learn how to lie.
BLEU: 1.0000

Input: hey, sweet cheeks.
Target: hi, joey.
Predicted: hi, joey.
BLEU: 1.0000

Input: joey never told you we went out, did he?
Target: what?
Predicted: what?
BLEU: 0.2214


Average BLEU Score on 100 samples: 0.6257


In [ ]:
print("Bot:", decode_sequence("hello"))
print("Bot:", decode_sequence("how are you?"))
print("Bot:", decode_sequence("what's your name?"))


Bot: no, i d like to say about it everything s the most important thing of me.
Bot: i m on the girl s
Bot: hammond.
